#### Info on fact table

In [0]:
%sql
DESCRIBE TABLE workspace.analytics.fact_sales;

col_name,data_type,comment
order_number,string,null
product_key,bigint,null
customer_key,bigint,null
order_date,date,null
shipping_date,date,null
due_date,date,null
sales_amount,bigint,null
quantity,bigint,null
price,bigint,null


#### Unique customer countries

In [0]:
%sql
SELECT country
FROM workspace.analytics.dim_customers
GROUP BY country;

country
Australia
United States
Canada
Germany
United Kingdom
France
n/a


#### Product categories, subcategories and products

In [0]:
%sql
SELECT category, subcategory, product_name
FROM workspace.analytics.dim_products
GROUP BY category, subcategory, product_name;

category,subcategory,product_name
Components,Road Frames,HL Road Frame - Black- 58
Components,Road Frames,HL Road Frame - Red- 58
Bikes,Mountain Bikes,Mountain-100 Black- 38
Bikes,Mountain Bikes,Mountain-100 Black- 42
Bikes,Mountain Bikes,Mountain-100 Black- 44
Bikes,Mountain Bikes,Mountain-100 Black- 48
Bikes,Mountain Bikes,Mountain-100 Silver- 38
Bikes,Mountain Bikes,Mountain-100 Silver- 42
Bikes,Mountain Bikes,Mountain-100 Silver- 44
Bikes,Mountain Bikes,Mountain-100 Silver- 48


#### Total sales, customers, products and orders

In [0]:
%sql
SELECT
  YEAR(order_date) AS year,
  SUM(sales_amount) AS total_sales,
  COUNT(DISTINCT customer_key) AS total_customers,
  COUNT(DISTINCT product_key) AS total_products,
  SUM(quantity) AS total_orders
FROM analytics.fact_sales
GROUP BY YEAR(order_date)
ORDER BY YEAR(order_date)

year,total_sales,total_customers,total_products,total_orders
null,4992,15,15,19
2010,43419,14,10,14
2011,7075088,2216,37,2216
2012,5842231,3255,80,3397
2013,16344878,17427,102,52807
2014,45642,834,42,1970


#### Sales by category and product

In [0]:
%sql
SELECT
    d.category,
    d.product_line,
    SUM(quantity) AS total_orders,
    FORMAT_NUMBER (SUM (sales_amount), 0) AS total_sales
FROM analytics.fact_sales AS f
LEFT JOIN analytics.dim_products AS d
    ON f.product_key = d.product_key 
GROUP BY d.category, d.product_line
ORDER BY d.category, d.product_line

category,product_line,total_orders,total_sales
Accessories,Mountain,10922,"227,398"
Accessories,Other Sales,15851,"340,009"
Accessories,Road,6916,"98,300"
Accessories,Touring,2423,"34,555"
Bikes,Mountain,4970,"9,952,254"
Bikes,Road,8068,"14,519,438"
Bikes,Touring,2167,"3,844,580"
Clothing,Mountain,1019,"71,330"
Clothing,Other Sales,7519,"263,274"
Clothing,Road,568,"5,112"


#### Revenue over time

In [0]:
%sql
SELECT
  DATE_FORMAT (order_date, 'yyyy-MM') AS date,
  SUM (sales_amount) AS total_sales
FROM analytics.fact_sales
GROUP BY date_format(order_date, 'yyyy-MM')
ORDER BY date_format(order_date, 'yyyy-MM')

date,total_sales
null,4992
2010-12,43419
2011-01,469795
2011-02,466307
2011-03,485165
2011-04,502042
2011-05,561647
2011-06,737793
2011-07,596710
2011-08,614516


Databricks visualization. Run in Databricks to view.

#### Total sales by month, Running total of sales and Moving average price over time

In [0]:
%sql
WITH sum AS (
  SELECT
    DATE_FORMAT(order_date, 'yyyy-MM') AS date,
    SUM (sales_amount) AS total_sales,
    AVG (price) AS avg_price
  FROM analytics.fact_sales
  WHERE order_date IS NOT NULL
  GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
)

SELECT 
  date,
  total_sales,
  SUM (total_sales) OVER (ORDER BY date) AS running_total_sales,
  ROUND (AVG (avg_price) OVER (ORDER BY date), 0) AS moving_avg_price
FROM sum
ORDER BY date



date,total_sales,running_total_sales,moving_avg_price
2010-12,43419,43419,3101.0
2011-01,469795,513214,3182.0
2011-02,466307,979521,3201.0
2011-03,485165,1464686,3209.0
2011-04,502042,1966728,3207.0
2011-05,561647,2528375,3210.0
2011-06,737793,3266168,3210.0
2011-07,596710,3862878,3205.0
2011-08,614516,4477394,3203.0
2011-09,603047,5080441,3209.0


#### Analyse the yearly performance of products by comparing their sales to both the average sales performance of the product and the previous year's sales

In [0]:
%sql
-- Yearly performance
WITH yearly AS (
SELECT
  d.product_name AS product_name,
  YEAR(f.order_date) AS year,
  SUM(f.sales_amount) AS total_sales_CY
FROM analytics.fact_sales AS f
LEFT JOIN analytics.dim_products AS d
  ON f.product_key = d.product_key
WHERE year(f.order_date) IS NOT NULL
GROUP BY d.product_name, YEAR(f.order_date)
)

SELECT
  product_name,
  year,
  total_sales_CY,
  -- Average sales performance
  AVG(total_sales_CY) OVER (PARTITION BY product_name ORDER BY year) AS avg_sales,

  -- Difference to Average
  total_sales_CY - AVG(total_sales_CY) OVER (PARTITION BY product_name ORDER BY year) AS diff_to_avg,
  CASE 
    WHEN diff_to_avg > 0 THEN 'Above Avg'
    WHEN diff_to_avg < 0 THEN 'Below Avg'
    ELSE 'Same as Avg'
  END AS avg_change,

  -- Previous year sales using Lag function
  LAG(total_sales_CY, 1) OVER (PARTITION BY product_name ORDER BY year ) AS total_sales_LY,

  -- Difference to LY
  total_sales_CY - LAG(total_sales_CY, 1) OVER (PARTITION BY product_name ORDER BY year) AS diff_to_LY,
  CASE
    WHEN diff_to_LY > 0 THEN 'Above LY'
    WHEN diff_to_LY < 0 THEN 'Below LY'
    ELSE 'Same as LY'
  END AS LY_change

FROM yearly
ORDER BY product_name, year

product_name,year,total_sales_CY,avg_sales,diff_to_avg,avg_change,total_sales_LY,diff_to_LY,LY_change
AWC Logo Cap,2012,72,72.0,0.0,Same as Avg,null,null,Same as LY
AWC Logo Cap,2013,18891,9481.5,9409.5,Above Avg,72,18819,Above LY
AWC Logo Cap,2014,747,6570.0,-5823.0,Below Avg,18891,-18144,Below LY
All-Purpose Bike Stand,2012,159,159.0,0.0,Same as Avg,null,null,Same as LY
All-Purpose Bike Stand,2013,37683,18921.0,18762.0,Above Avg,159,37524,Above LY
All-Purpose Bike Stand,2014,1749,13197.0,-11448.0,Below Avg,37683,-35934,Below LY
Bike Wash - Dissolver,2013,6960,6960.0,0.0,Same as Avg,null,null,Same as LY
Bike Wash - Dissolver,2014,312,3636.0,-3324.0,Below Avg,6960,-6648,Below LY
Classic Vest- L,2013,11968,11968.0,0.0,Same as Avg,null,null,Same as LY
Classic Vest- L,2014,512,6240.0,-5728.0,Below Avg,11968,-11456,Below LY


#### Segment products into cost ranges and count how many products fall into each segment

In [0]:
%sql
WITH cost_segment AS (
  SELECT
    product_key,
    product_name, 
    cost,
    CASE
      WHEN cost < 100 THEN 'Below 100'
      WHEN cost BETWEEN 100 AND 500 THEN '100-500'
      WHEN cost BETWEEN 500 AND 1000 THEN '500-1000'
      WHEN cost > 1000 THEN 'Above 1000'
    END AS cost_range
  FROM analytics.dim_products
)

SELECT
  cost_range,
  COUNT(DISTINCT product_key) AS number_of_products
FROM cost_segment
GROUP BY cost_range
ORDER BY number_of_products DESC

cost_range,number_of_products
Below 100,110
100-500,101
500-1000,45
Above 1000,39


#### Group customers into three segments based on their spending behavior:
- VIP: Customers with at least 12 months of history and spending more than €5,000.
- Regular: Customers with at least 12 months of history but spending €5,000 or less.
- New: Customers with a lifespan less than 12 months.
- And find the total number of customers by each group

In [0]:
%sql
WITH customer_history AS (
  SELECT
      d.customer_key AS customer_key,
      SUM(f.sales_amount) AS total_spend,
      MIN(f.order_date) AS first_date,
      MAX(f.order_date) AS last_date,
      DATEDIFF(MAX(f.order_date),MIN(f.order_date)) AS lifespan
  FROM analytics.fact_sales AS f
  LEFT JOIN analytics.dim_customers AS d
      ON f.customer_key = d.customer_key
  WHERE f.order_date IS NOT NULL
  GROUP BY d.customer_key
)

SELECT
  customer_segment,
  COUNT(DISTINCT customer_key) AS total_customers
FROM (
  SELECT
      customer_key,
      CASE
          WHEN lifespan >= 12 AND total_spend <= 5000 THEN 'Regular'
          WHEN lifespan >= 12 AND total_spend > 5000 THEN 'VIP'
          ELSE 'New'
      END AS customer_segment
  FROM customer_history
) AS segmented_customers
GROUP BY customer_segment
ORDER BY total_customers DESC;

customer_segment,total_customers
New,11709
Regular,5043
VIP,1730


Databricks visualization. Run in Databricks to view.

#### Which categories contribute the most to overall sales

In [0]:
%sql
WITH category_sales AS (
  SELECT 
    d.category,
    SUM(f.sales_amount) AS total_sales
  FROM analytics.fact_sales f
  LEFT JOIN analytics.dim_products d
    ON f.product_key = d.product_key
  GROUP BY d.category
)

SELECT 
 category,
 ROUND((CAST (total_sales AS FLOAT) / SUM(total_sales) OVER()) * 100, 2) AS pct_sales
FROM category_sales
ORDER BY pct_sales DESC

category,pct_sales
Bikes,96.46
Accessories,2.39
Clothing,1.16


Databricks visualization. Run in Databricks to view.